# DCMC-Fusion — K = 3

## Experimental stage 3 (article, Section 4.4)

This standalone notebook evaluates all **10 combinations** of 3 architectures. Each architecture contributes four spectral probabilities (Mel, LOG, DWT, CQT), giving **12 input features** and **40 classifier configurations**.

Labels: 0 = natural; 1 = synthetic. Probabilities are percentages in [0, 100]. The column order follows the original notebook.

Run cells in order. The original protocol is retained: stratified 70/30 split, random_state=42, StandardScaler inside a Pipeline, five-fold GridSearchCV scored by macro F1, and the original grids for LR, RF, XGBoost and SVM. Training can take substantial time. No experiment outputs are precomputed.

## 1. Dependencies
Run the installation cell once if the packages are unavailable.

In [ ]:
%pip install numpy pandas scikit-learn xgboost ipython --quiet

In [ ]:
from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
np.random.seed(42)

## 2. Architecture configuration and input data

In [ ]:
K = 3
ARCHITECTURES = {'resnet50': ['resnet50_mel', 'resnet50_log', 'resnet50_dwt', 'resnet50_cqt'], 'densenet121': ['densenet121_mel', 'densenet121_log', 'densenet121_dwt', 'densenet121_cqt'], 'inception_v3': ['inception_v3_mel', 'inception_v3_log', 'inception_v3_dwt', 'inception_v3_cqt'], 'resnet34': ['resnet34_mel', 'resnet34_log', 'resnet34_dwt', 'resnet34_cqt'], 'vgg16': ['vgg16_mel', 'vgg16_log', 'vgg16_dwt', 'vgg16_cqt']}
ARCH_ORDER = ['resnet50', 'densenet121', 'inception_v3', 'resnet34', 'vgg16']

In [ ]:
# Place totalResultados.csv beside this notebook (or upload it to /content in Colab).
CSV_PATH = Path("totalResultados.csv")
if not CSV_PATH.is_file() and Path("/content/totalResultados.csv").is_file():
    CSV_PATH = Path("/content/totalResultados.csv")
df = pd.read_csv(CSV_PATH)
required = ["audio", "label"] + [col for cols in ARCHITECTURES.values() for col in cols]
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")
values = df[[col for cols in ARCHITECTURES.values() for col in cols]].to_numpy(dtype=float)
if not np.isfinite(values).all() or ((values < 0) | (values > 100)).any():
    raise ValueError("Probabilities must be finite and in [0, 100].")
if set(df["label"].unique()) != {0, 1}:
    raise ValueError("Labels must contain 0 (natural) and 1 (synthetic).")
y = df["label"].to_numpy()
print(f"Audio samples: {len(df)}")
display(df["label"].value_counts().rename_axis("label").to_frame("samples"))
CONFIGURATIONS = list(combinations(ARCH_ORDER, K))
display(pd.DataFrame({"configuration": [" + ".join(c) for c in CONFIGURATIONS],
                      "features": [4 * K] * len(CONFIGURATIONS)}))

## 3. Fusion classifiers and hyperparameter search

In [ ]:
ML_MODELS = {'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), {'model__C': [0.01, 0.1, 1, 10], 'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear', 'saga']}), 'Random Forest': (RandomForestClassifier(random_state=42, n_jobs=-1), {'model__n_estimators': [100, 300, 700], 'model__max_depth': [3, 5, 10, None], 'model__min_samples_split': [3, 5, 8]}), 'XGBoost': (XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0), {'model__n_estimators': [100, 300, 700], 'model__learning_rate': [0.01, 0.1, 1], 'model__max_depth': [3, 5, 8]}), 'SVM': (SVC(probability=True, random_state=42), {'model__C': [0.1, 1, 5, 10], 'model__gamma': ['scale', 'auto'], 'model__kernel': ['rbf']})}

In [ ]:
def entrenar(X_raw, y):
    X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.3, random_state=42, stratify=y)
    resultados = {}
    for nombre, (modelo, params) in ML_MODELS.items():
        pipeline = Pipeline([('scaler', StandardScaler()), ('model', modelo)])
        gs = GridSearchCV(estimator=pipeline, param_grid=params, cv=5, scoring='f1_macro', n_jobs=-1, refit=True)
        gs.fit(X_train, y_train)
        y_pred = gs.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, target_names=['natural', 'sintetico'], output_dict=True)
        resultados[nombre] = {'y_test': y_test, 'y_pred': y_pred, 'accuracy': accuracy, 'report': report, 'best_params': gs.best_params_, 'best_score': gs.best_score_, 'best_model': gs.best_estimator_}
    return resultados

## 4. Run all combinations for this K

In [ ]:
# Each configuration is trained once; its results are reused in all tables.
results = {}
for selected in CONFIGURATIONS:
    name = " + ".join(selected)
    columns = [column for architecture in selected for column in ARCHITECTURES[architecture]]
    print(f"K={K}: {name} ({len(columns)} features)", flush=True)
    results[name] = entrenar(df[columns].to_numpy(), y)
print(f"Completed: {len(results) * len(ML_MODELS)} classifier configurations.")

## 5. Global metrics, per-class metrics and best parameters
All metric values are in [0, 1]. Accuracy is a global measure; per-class tables report precision, recall, F1 and support.

In [ ]:
global_rows, class_rows, parameter_rows = [], [], []
for configuration, classifiers in results.items():
    for classifier, result in classifiers.items():
        base = {"K": K, "architectures": configuration, "classifier": classifier}
        macro = result["report"]["macro avg"]
        global_rows.append({**base, "accuracy": result["accuracy"],
                            "precision_macro": macro["precision"], "recall_macro": macro["recall"],
                            "f1_macro": macro["f1-score"]})
        for label in ["natural", "sintetico"]:
            metrics = result["report"][label]
            class_rows.append({**base, "class": label, "precision": metrics["precision"],
                               "recall": metrics["recall"], "f1": metrics["f1-score"],
                               "support": metrics["support"]})
        parameter_rows.append({**base, "best_cv_f1_macro": result["best_score"],
                               "best_params": json.dumps(result["best_params"], sort_keys=True)})
global_metrics = pd.DataFrame(global_rows)
class_metrics = pd.DataFrame(class_rows)
best_parameters = pd.DataFrame(parameter_rows)
display(global_metrics)
display(class_metrics)
display(best_parameters)

## 6. Export results

In [ ]:
OUTPUT_DIR = Path("results") / f"K{K}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
global_metrics.to_csv(OUTPUT_DIR / "global_metrics.csv", index=False)
class_metrics.to_csv(OUTPUT_DIR / "class_metrics.csv", index=False)
best_parameters.to_csv(OUTPUT_DIR / "best_parameters.csv", index=False)
print(f"Results saved to: {OUTPUT_DIR.resolve()}")